# Match ND-LAr (SPINE) and TMS true vertices, then view their hits

Matching follows `match_spine_to_truth_spill.py`:
1. **Position**: Truth_Spill `TrueVtxX/Y/Z` vs SPINE `ixn.vertex` (cm → mm), within `VERTEX_TOL_MM`. This is the reference.
2. **IDs**: SPINE `orig_id` is not the Truth_Spill vertex ID, so interactions are matched by their (track ID, PDG) signature with an order-preserving alignment. Used as a cross-check.

Run inside the `spine-nb` container (`/data` = `~/Downloads`).

In [1]:
import numpy as np
import pandas as pd
import uproot
import yaml
import plotly.graph_objects as go
from spine.driver import Driver

In [ ]:
data_dir = "/data/"
spine_file = data_dir + "MiniProdN5p3_NDComplex_FHC.spine.full.sanddrift.0000005.MLRECO_SPINE.hdf5"
tms_file = data_dir + "MiniProdN5p3_NDComplex_FHC.tmsreco.full.sanddrift.v4.0000005.TMSRECO.root"

VERTEX_TOL_MM = 1.0
THROUGHGOING_FLAG = 10**15  # orig_id flag on external (throughgoing) interactions

## Colour palette

In [ ]:
NDLAr_colour = '#e63946'

strong_colours = ['#E69F00', '#0072B2', '#009E73', '#CC79A7', '#D55E00', '#56B4E9', '#F0E442']
light_colours = ['#f0c975', '#70b1d9', '#7ecdb9', '#e0adc9', '#e8a176', '#a2d9f7', '#f7f2a1']

## Load files

In [ ]:
tms_root = uproot.open(tms_file)
tms = tms_root["Truth_Spill"].arrays(
    ["SpillNo", "TrueVtxGlobalID", "TrueVtxX", "TrueVtxY", "TrueVtxZ",
     "VertexGlobalID", "TrackId", "PDG"], library="np")

# TMS true hits live in Truth_Info (one entry per slice), tagged by global vertex ID
tms_hits = tms_root["Truth_Info"].arrays(
    ["SpillNo", "TrueHitX", "TrueHitY", "TrueHitZ", "TrueHitVertexId"], library="np")

cfg = f'''
base:
  verbosity: warning
build:
  mode: both
  fragments: false
  particles: true
  interactions: true
  units: cm
io:
  reader:
    file_keys: {spine_file}
    skip_unknown_attrs: True
    name: hdf5
'''
driver = Driver(yaml.safe_load(cfg))
spine_events = [driver.process(i) for i in range(len(driver))]
print(f"{len(tms['SpillNo'])} Truth_Spill entries, {len(spine_events)} SPINE events")

## Method 1: match by vertex position

In [ ]:
def tms_vertex_xyz(entry):
    return np.stack([tms["TrueVtxX"][entry], tms["TrueVtxY"][entry], tms["TrueVtxZ"][entry]], axis=1)


def match_by_position(tol_mm=VERTEX_TOL_MM):
    """{(SPINE event, ixn id): (Truth_Spill entry, global vertex ID, distance mm)}"""
    tms_xyz = [tms_vertex_xyz(e) for e in range(len(tms["SpillNo"]))]
    out = {}
    for ev, data in enumerate(spine_events):
        for ixn in data["truth_interactions"]:
            pos = np.asarray(ixn.vertex) * 10.0  # cm -> mm
            best = None
            for entry, xyz in enumerate(tms_xyz):
                dist = np.linalg.norm(xyz - pos, axis=1)
                j = int(dist.argmin())
                if dist[j] <= tol_mm and (best is None or dist[j] < best[2]):
                    best = (entry, int(tms["TrueVtxGlobalID"][entry][j]), float(dist[j]))
            if best is not None:
                out[(ev, int(ixn.id))] = best
    return out


position_matches = match_by_position()
print(f"{len(position_matches)} SPINE interactions matched to a Truth_Spill vertex within {VERTEX_TOL_MM} mm")

## Method 2: match by (track ID, PDG) signature + order-preserving alignment

In [ ]:
def tms_signatures(entry):
    """Per Truth_Spill vertex (sorted by global ID): (global ID, set of (track, pdg))."""
    gids = np.sort(tms["TrueVtxGlobalID"][entry])
    sig = {int(g): set() for g in gids}
    for g, trk, pdg in zip(tms["VertexGlobalID"][entry], tms["TrackId"][entry], tms["PDG"][entry]):
        g = int(g) % THROUGHGOING_FLAG
        if g in sig:
            sig[g].add((int(trk), int(pdg)))
    return [(int(g), sig[int(g)]) for g in gids]


def spine_signatures(data):
    """Non-throughgoing SPINE interactions in orig_id order: (ixn, set of (track, pdg))."""
    ixns = sorted((x for x in data["truth_interactions"] if int(x.orig_id) < THROUGHGOING_FLAG),
                  key=lambda x: int(x.orig_id))
    sig = {x.id: set() for x in ixns}
    for p in data["truth_particles"]:
        if p.interaction_id in sig:
            sig[p.interaction_id].add((int(p.track_id), int(p.pdg_code)))
    return [(x, sig[x.id]) for x in ixns]


def align(spine_sig, tms_sig):
    """Order-preserving assignment; returns (total score, {spine position: tms position})."""
    n, m = len(spine_sig), len(tms_sig)
    score = np.zeros((n, m))
    for i, (_, s) in enumerate(spine_sig):
        for j, (_, t) in enumerate(tms_sig):
            score[i, j] = (len(t & s) - 0.5 * len(t - s)) / max(len(t), 1)
    best = np.full((n + 1, m + 1), -1e9)
    best[0, :] = 0
    took = np.zeros((n + 1, m + 1), bool)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            take = best[i - 1, j - 1] + score[i - 1, j - 1]
            best[i, j] = max(best[i, j - 1], take)
            took[i, j] = take >= best[i, j - 1]
    i, j, assign = n, m, {}
    while i > 0 and j > 0:
        if took[i, j]:
            assign[i - 1] = j - 1
            i, j = i - 1, j - 1
        else:
            j -= 1
    return best[n, m], assign


def match_by_ids():
    """{(SPINE event, ixn id): (Truth_Spill entry, global vertex ID)}"""
    tms_sigs = [tms_signatures(e) for e in range(len(tms["SpillNo"]))]
    out = {}
    for ev, data in enumerate(spine_events):
        s_sig = spine_signatures(data)
        results = [align(s_sig, t_sig) for t_sig in tms_sigs]
        entry = int(np.argmax([r[0] for r in results]))
        for i, j in results[entry][1].items():
            out[(ev, int(s_sig[i][0].id))] = (entry, tms_sigs[entry][j][0])
    return out


id_matches = match_by_ids()
print(f"{len(id_matches)} SPINE interactions matched by IDs")

## Match table

In [ ]:
rows = []
for (ev, ixn_id), (entry, gid, dist) in sorted(position_matches.items()):
    ixn = spine_events[ev]["truth_interactions"][ixn_id]
    id_match = id_matches.get((ev, ixn_id))
    n_tms_hits = sum(int(np.sum(v == gid)) for s, v in zip(tms_hits["SpillNo"], tms_hits["TrueHitVertexId"])
                     if s == tms["SpillNo"][entry])
    rows.append({
        "SPINE_event": ev,
        "ixn_id": ixn_id,
        "orig_id": int(ixn.orig_id),
        "TS_entry": entry,
        "TS_vtx_global_id": gid,
        "dist_mm": round(dist, 4),
        "id_method_agrees": None if id_match is None else id_match == (entry, gid),
        "n_NDLAr_points": len(ixn.points),
        "n_TMS_hits": n_tms_hits,
    })
match_df = pd.DataFrame(rows)
print("ID method agreement:", match_df["id_method_agrees"].value_counts(dropna=False).to_dict())
pd.set_option("display.max_rows", 50)
match_df

In [ ]:
# Vertices with hits in both detectors are the interesting ones for track matching
both = match_df[(match_df.n_NDLAr_points > 0) & (match_df.n_TMS_hits > 0)]
print(f"{len(both)} matched vertices have hits in both ND-LAr and TMS")
both

## Figure setup and detector boxes

In [ ]:
def plot_box(fig, x_min, x_max, y_min, y_max, z_min, z_max, color, name):
    # One continuous path over every edge; None breaks the line between disconnected edges
    x = [x_min, x_max, x_max, x_min, x_min, x_min, x_max, x_max, x_min, x_min, None, x_max, x_max, None, x_min, x_min, None, x_max, x_max]
    y = [y_min, y_min, y_max, y_max, y_min, y_min, y_min, y_max, y_max, y_min, None, y_min, y_min, None, y_max, y_max, None, y_max, y_max]
    z = [z_min, z_min, z_min, z_min, z_min, z_max, z_max, z_max, z_max, z_max, None, z_min, z_max, None, z_min, z_max, None, z_min, z_max]
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode="lines",
                               line=dict(color=color, width=3), name=name, showlegend=True))


def setup_figure(figure, title):
    figure.update_layout(
        width=1600, height=900,
        scene=dict(
            xaxis=dict(title="x [mm]", range=[-4500, 4500]),
            yaxis=dict(title="y [mm]", range=[-4000, 1000]),
            zaxis=dict(title="z [mm]", range=[0, 20000]),
            camera=dict(eye=dict(x=2.0, y=4.0, z=0.0), up=dict(x=0, y=-1, z=0)),  # Y is up
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=4),
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        showlegend=True,
        legend=dict(x=0.02, y=0.98, xanchor='left', yanchor='top',
                    bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='Black', borderwidth=1,
                    itemsizing='constant'),
        title=dict(text=title, x=0.5, xanchor='center'),
    )
    figure.data = []
    # Geometry from TMS_Constants.h (mm)
    plot_box(figure, -3730, 3730, -3702.23, 997.77, 11177.5, 18502.5, "blue", "TMS detector")
    plot_box(figure, -3478.48, 3478.48, -2166.71, 829.282, 4179.24, 9135.88, "black", "ND-LAr detector")

## Plot hits for matched vertices

In [ ]:
def plot_matched_vertex(figure, row, colour_idx):
    strong = strong_colours[colour_idx % len(strong_colours)]
    light = light_colours[colour_idx % len(light_colours)]
    ev, ixn_id, entry, gid = int(row.SPINE_event), int(row.ixn_id), int(row.TS_entry), int(row.TS_vtx_global_id)
    label = f"(ev {ev} ixn {ixn_id} / vtx {gid})"

    # ND-LAr hits and vertex from SPINE (cm -> mm)
    ixn = spine_events[ev]["truth_interactions"][ixn_id]
    pts = np.asarray(ixn.points) * 10.0
    if len(pts):
        figure.add_trace(go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers",
            marker=dict(size=2, color=strong, symbol='circle'),
            name=f"{label} ND-LAr hits", hoverlabel_namelength=-1))
    v = np.asarray(ixn.vertex) * 10.0
    figure.add_trace(go.Scatter3d(
        x=[v[0]], y=[v[1]], z=[v[2]], mode="markers",
        marker=dict(size=5, color=NDLAr_colour, symbol='square-open'),
        name=f"{label} SPINE vertex [{v[0]:.1f}, {v[1]:.1f}, {v[2]:.1f}]", hoverlabel_namelength=-1))

    # TMS true hits from all Truth_Info slices in this spill tagged with this vertex
    spill_no = tms["SpillNo"][entry]
    hx, hy, hz = [], [], []
    for s, x, y, z, vid in zip(tms_hits["SpillNo"], tms_hits["TrueHitX"], tms_hits["TrueHitY"],
                               tms_hits["TrueHitZ"], tms_hits["TrueHitVertexId"]):
        if s != spill_no:
            continue
        sel = vid == gid
        hx.extend(x[sel]); hy.extend(y[sel]); hz.extend(z[sel])
    if hx:
        figure.add_trace(go.Scatter3d(
            x=hx, y=hy, z=hz, mode="markers",
            marker=dict(size=2, color=light, symbol='circle'),
            name=f"{label} TMS true hits ({len(hx)})", hoverlabel_namelength=-1))

    # Truth_Spill vertex
    k = int(np.where(tms["TrueVtxGlobalID"][entry] == gid)[0][0])
    tx, ty, tz = tms["TrueVtxX"][entry][k], tms["TrueVtxY"][entry][k], tms["TrueVtxZ"][entry][k]
    figure.add_trace(go.Scatter3d(
        x=[tx], y=[ty], z=[tz], mode="markers",
        marker=dict(size=5, color='black', symbol='cross'),
        name=f"{label} Truth_Spill vertex [{tx:.1f}, {ty:.1f}, {tz:.1f}]", hoverlabel_namelength=-1))

In [ ]:
def draw(rows):
    # Plot the matched vertices in the given match_df rows
    fig = go.Figure()
    setup_figure(fig, f"Matched vertices (match_df rows): {[int(r) for r in rows]}")
    for c, r in enumerate(rows):
        plot_matched_vertex(fig, match_df.loc[r], c)
    fig.show()

In [ ]:
# Choose match_df rows to plot (default: first few vertices with hits in both detectors)
rows_to_plot = list(both.index[:3]) if len(both) else [0]
draw(rows_to_plot)